In [ ]:
import os
import bpy
import math
import numpy as np
import open3d as o3d
import plotly.graph_objects as go

from pathlib import Path

In [ ]:
# bpy.ops.wm.open_mainfile(filepath="/home/addai/Blender/master_chief.blend")
bpy.ops.wm.open_mainfile(filepath="/home/addai/Blender/Perseverance_Rover.blend")

In [ ]:
from mathutils import Matrix, Vector


def set_sun_direction(sun, azimuth_deg, elevation_deg):
    azimuth = np.radians(azimuth_deg)
    elevation = np.radians(elevation_deg)
    x = np.cos(elevation) * np.cos(azimuth)
    y = np.cos(elevation) * np.sin(azimuth)
    z = np.sin(elevation)
    direction = Vector((x, y, z))
    rot_quat = direction.to_track_quat("-Z", "Y")
    sun.rotation_euler = rot_quat.to_euler()

In [ ]:
sun = bpy.data.objects["Sun"]
print(sun.rotation_euler)
print(sun.location)

In [ ]:
from mathutils import Euler

sun.rotation_euler = Euler((-0.0534, -0.6822, 1.9356))
print(sun.rotation_euler)

## Depth rendering


In [ ]:
scene = bpy.context.scene
scene.use_nodes = True
tree = scene.node_tree
tree.nodes.clear()

bpy.context.view_layer.use_pass_z = True

# Render Layers node
rl = tree.nodes.new("CompositorNodeRLayers")

In [ ]:
g_depth_color_mode = "BW"
g_depth_color_depth = "16"
g_depth_file_format = "PNG"
g_depth_clip_start = 0.5
g_depth_clip_end = 4

links = tree.links

for node in tree.nodes:
    tree.nodes.remove(node)

render_layer_node = tree.nodes.new("CompositorNodeRLayers")
map_value_node = tree.nodes.new("CompositorNodeMapValue")
file_output_node = tree.nodes.new("CompositorNodeOutputFile")

map_value_node.offset[0] = -g_depth_clip_start
map_value_node.size[0] = 1 / (g_depth_clip_end - g_depth_clip_start)
map_value_node.use_min = True
map_value_node.use_max = True
map_value_node.min[0] = 0.0
map_value_node.max[0] = 1.0

file_output_node.format.color_mode = g_depth_color_mode
file_output_node.format.color_depth = g_depth_color_depth
file_output_node.format.file_format = g_depth_file_format
file_output_node.base_path = "."

links.new(render_layer_node.outputs[2], map_value_node.inputs[0])
links.new(map_value_node.outputs[0], file_output_node.inputs[0])

In [ ]:
file_output_node = bpy.context.scene.node_tree.nodes[2]
file_output_node.file_slots[0].path = "blender-######.depth.png"  # blender placeholder #

bpy.ops.render.render(write_still=True)

Normalized depth png


In [ ]:
normalize = tree.nodes.new("CompositorNodeNormalize")
tree.links.new(rl.outputs["Depth"], normalize.inputs[0])

# Output File node for PNG
depth_output = tree.nodes.new("CompositorNodeOutputFile")
depth_output.base_path = "."
depth_output.format.file_format = "PNG"
depth_output.format.color_mode = "BW"
depth_output.format.color_depth = "16"  # 16-bit PNG

tree.links.new(normalize.outputs[0], depth_output.inputs[0])
depth_output.file_slots[0].path = "depth_"

bpy.ops.render.render(write_still=True)

Metric depth


In [ ]:
depth_output = tree.nodes.new("CompositorNodeOutputFile")
depth_output.base_path = "."
depth_output.format.file_format = "PNG"
depth_output.format.color_depth = "16"  # 32-bit float per channel
depth_output.format.color_mode = "BW"  # Single channel

tree.links.new(rl.outputs["Depth"], depth_output.inputs[0])
depth_output.file_slots[0].path = "depth_####"

bpy.ops.render.render(write_still=True)

In [ ]:
import imageio.v3 as iio

depth_exr = iio.imread("depth_0001.exr")  # shape: (H, W, 3)
depth = depth_exr[..., 0]  # All channels are equal; use any

In [ ]:
depth.dtype

In [ ]:
from PIL import Image

im = Image.open("blender-000001.depth.png")
depth = np.array(im)
print(depth.shape)
print(depth.dtype)
print(depth)

In [ ]:
np.unique(depth)

In [ ]:
# Render the image
bpy.ops.render.render(write_still=False)

In [ ]:
# Set up rendering of depth map:
bpy.context.scene.use_nodes = True
tree = bpy.context.scene.node_tree
links = tree.links

# clear default nodes
for n in tree.nodes:
    tree.nodes.remove(n)

# create input render layer node
rl = tree.nodes.new("CompositorNodeRLayers")

map = tree.nodes.new(type="CompositorNodeMapValue")
# Size is chosen kind of arbitrarily, try out until you're satisfied with resulting depth map.
map.size = [0.08]
map.use_min = True
map.min = [0]
map.use_max = True
map.max = [255]
links.new(rl.outputs[2], map.inputs[0])

invert = tree.nodes.new(type="CompositorNodeInvert")
links.new(map.outputs[0], invert.inputs[1])

# The viewer can come in handy for inspecting the results in the GUI
depthViewer = tree.nodes.new(type="CompositorNodeViewer")
links.new(invert.outputs[0], depthViewer.inputs[0])
# Use alpha from input.
links.new(rl.outputs[1], depthViewer.inputs[1])

In [ ]:
# create a file output node and set the path
fileOutput = tree.nodes.new(type="CompositorNodeOutputFile")
fileOutput.base_path = "results/temp"
links.new(invert.outputs[0], fileOutput.inputs[0])

In [ ]:
from PIL import Image

im = Image.open("results/temp/Image0001.png")
depth = np.array(im)
print(depth.shape, depth.dtype)

In [ ]:
depth

In [ ]:
depth_map = bpy.data.images["Render Result"].pixels
depth_map[:]

In [ ]:
scene = bpy.context.scene

# Enable use of nodes and depth pass
scene.use_nodes = True
tree = scene.node_tree
tree.nodes.clear()

# Create Render Layers node
rl = tree.nodes.new(type="CompositorNodeRLayers")

# Create Output node for depth
depth_output = tree.nodes.new(type="CompositorNodeOutputFile")
depth_output.label = "Depth Output"
depth_output.base_path = "/tmp/"  # Change to your desired output folder

# Connect depth output
tree.links.new(rl.outputs["Depth"], depth_output.inputs[0])

# Optional: Set file format to OpenEXR for floating-point depth
depth_output.format.file_format = "OPEN_EXR"

# Set render settings
scene.render.filepath = "/tmp/color.png"  # Color output
bpy.context.scene.render.image_settings.file_format = "PNG"

# Render
bpy.ops.render.render(write_still=True)

In [ ]:
light_obj = bpy.data.objects["Sun"]

# Now you can access the pose:
print(light_obj.matrix_world)
print(light_obj.location)
print(light_obj.rotation_euler)

In [ ]:
print(np.array(light_obj.matrix_world))

In [ ]:
light = bpy.data.lights["Sun"]
light.matrix_world

In [ ]:
scene = bpy.context.scene

resolution_x = scene.render.resolution_x
resolution_y = scene.render.resolution_y
scale = scene.render.resolution_percentage / 100
width = resolution_x * scale
height = resolution_y * scale

In [ ]:
width

In [ ]:
camera = bpy.data.objects["Camera"]
cam_data = camera.data

In [ ]:
sensor_width_mm = cam_data.sensor_width  # in mm (typically 36 for full-frame)
focal_length_mm = cam_data.lens  # in mm

focal_x = (focal_length_mm / sensor_width_mm) * width
focal_y = (
    focal_x * (height / width)
    if cam_data.sensor_fit != "VERTICAL"
    else (focal_length_mm / cam_data.sensor_height) * height
)

In [ ]:
target_height, target_width = 1024, 1024

camera_angle_x = 0.9500215649604797
focal_length = 0.5 * width / np.tan(0.5 * camera_angle_x)
focal_length = focal_length * target_width / width
fx = focal_length
fy = focal_length
cx = target_width / 2.0
cy = target_height / 2.0

In [ ]:
print(cam_data.sensor_width)
print(cam_data.sensor_height)

In [ ]:
scene.render.resolution_x

# Sample points from mesh


In [ ]:
from shadow_splat.util.general import generate_ply_from_points

In [ ]:
def blender_mesh_to_open3d(obj_name: str) -> o3d.geometry.TriangleMesh:
    # Get the Blender object
    obj = bpy.data.objects[obj_name]
    mesh = obj.to_mesh()
    mesh.calc_loop_triangles()

    # Transform vertices to world space
    vertices = np.array([obj.matrix_world @ v.co for v in mesh.vertices])

    # Collect triangle indices
    triangles = []
    for tri in mesh.loop_triangles:
        triangles.append([tri.vertices[0], tri.vertices[1], tri.vertices[2]])
    triangles = np.array(triangles)

    # Create Open3D triangle mesh
    o3d_mesh = o3d.geometry.TriangleMesh()
    o3d_mesh.vertices = o3d.utility.Vector3dVector(vertices)
    o3d_mesh.triangles = o3d.utility.Vector3iVector(triangles)
    o3d_mesh.compute_vertex_normals()

    return o3d_mesh

In [ ]:
body = bpy.data.objects["Body"]
child_points = []
for child in body.children:
    mesh = blender_mesh_to_open3d(child.name)
    points = mesh.sample_points_poisson_disk(10000)
    points = np.asarray(points.points)
    child_points.append(points)

In [ ]:
child_points = np.concatenate(child_points, axis=0)
child_points.shape

In [ ]:
child_colors = 120 * np.ones_like(child_points)
generate_ply_from_points(child_points, child_colors, "rover_children.ply")

In [ ]:
body_mesh = blender_mesh_to_open3d("Body")

In [ ]:
body_points = body_mesh.sample_points_poisson_disk(100000)
# body_points = body_mesh.sample_points_uniformly(1000000)
body_points = np.asarray(body_points.points)
body_colors = 120 * np.ones_like(body_points)

In [ ]:
from shadow_splat.util.general import generate_ply_from_points

generate_ply_from_points(body_points, body_colors, "rover_body.ply")

In [ ]:
plane = blender_mesh_to_open3d("Plane")
plane_points = plane.sample_points_poisson_disk(100000)
# plane_points = plane.sample_points_uniformly(1000000)
plane_points = np.asarray(plane_points.points)
plane_colors = 255 * np.ones_like(plane_points)

# generate_ply_from_points(plane_points, plane_colors, "rover_plane.ply")

In [ ]:
all_points = np.concatenate([plane_points, child_points, body_points], axis=0)
all_colors = np.concatenate([plane_colors, child_colors, body_colors])

generate_ply_from_points(all_points, all_colors, "rover_all.ply")

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=all_points[:, 0],
        y=all_points[:, 1],
        z=all_points[:, 2],
        mode="markers",
        marker=dict(size=2, color="blue"),
    )
)
fig.show()

In [ ]:
body = bpy.data.objects["Body"]
mesh = body.to_mesh()
mesh.calc_loop_triangles()

In [ ]:
# List all mesh objects
for obj in bpy.data.objects:
    if obj.type == "MESH":
        print(f"Mesh: {obj.name}, Vertices: {len(obj.data.vertices)}")

In [ ]:
mesh.loop_triangles

In [ ]:
bpy.context.scene.render.engine = "CYCLES"
bpy.context.scene.render.resolution_x = 1024
bpy.context.scene.render.resolution_y = 1024
bpy.context.scene.cycles.samples = 128
# bpy.context.scene.render.image_settings.color_mode = "BW"
# bpy.context.scene.render.image_settings.color_depth = "8"
# bpy.context.scene.render.image_settings.color_depth = "8"

In [ ]:
bpy.data.objects["Camera"]

In [ ]:
from mathutils import Matrix


def set_camera_pose(cam, location, target):
    direction = (target[0] - location[0], target[1] - location[1], target[2] - location[2])
    direction = np.array(direction)  # This is -z
    direction = direction / np.linalg.norm(direction)

    up = np.array([0, 0, 1])

    right = np.cross(up, -direction)
    up = np.cross(-direction, right)

    matrix_world = np.eye(4)
    rotation = np.stack([right, up, -direction], axis=-1)
    translation = location
    matrix_world[:3, :3] = rotation
    matrix_world[:3, -1] = translation
    cam.matrix_world = Matrix(matrix_world)

    return matrix_world

In [ ]:
# Set output path and format
bpy.context.scene.render.image_settings.file_format = "PNG"  # or 'JPEG', etc.
bpy.context.scene.render.filepath = "rendered_image.png"

# Set camera position based on azimuth and elevation


# # Set camera parameters
# azimuth = 45  # degrees
# elevation = 30  # degrees
# distance = 5  # distance from origin

# # Convert spherical to Cartesian coordinates
# x = distance * math.cos(math.radians(elevation)) * math.cos(math.radians(azimuth))
# y = distance * math.cos(math.radians(elevation)) * math.sin(math.radians(azimuth))
# z = distance * math.sin(math.radians(elevation))

# # Position camera
# camera = bpy.data.objects["Camera"]
# camera.location = (x, y, z)

# # Point camera at origin
# direction = camera.location
# rot_quat = direction.to_track_quat('-Z', 'Y')
# camera.rotation_euler = rot_quat.to_euler()

# # Set as active camera
# bpy.context.scene.camera = camera

camera = bpy.data.objects["Camera"]


# Render the image and save it to disk
bpy.ops.render.render(write_still=True)